In [1]:
# The codes in this notebook is partially based on (https://github.com/juliansester/nga/blob/main/Example%20SP500-Asian.ipynb).
import torch
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from GAD_util import *

import pandas as pd
from scipy.optimize import minimize
import os

In [2]:
def compute_max_parameters(x, 
                           iterations = 1000,
                           initial_guess= [0.1,0.1,0,0,0.75],
                           tolerance = 1e-15,
                           Delta = 1/250.,
                           method = 'COBYLA'):
    x_0 = np.array(initial_guess) #Initial guess
    eps = tolerance # Tolerance to avoid that fractions and log-expressions become inf or -inf
    
    #Definte the Log-Likelihood Function
    def log_likelihood(param):
        a_0 = param[0]
        a_1 = param[1]
        b_0 = param[2]
        b_1 = param[3]
        gamma = param[4]
        constant = np.sqrt(2*np.pi*Delta)
        l= [-np.log((a_0+a_1*np.maximum(x[i],0))**gamma*constant+eps)-(1/(2*Delta))*((x[i+1]-x[i]-(b_0+b_1*x[i])*Delta)/(a_0+a_1*np.maximum(x[i],0)+eps)**gamma)**2 for i in range(len(x)-1)]
        return -np.mean(l) # Mean instead of sum to have smaller values
    
    a0,a1,b0,b1,gamma = minimize(log_likelihood,x_0,method=method,options={'maxiter': iterations,
                                                                          'rhobeg':0.01},
                bounds = [(eps,None),(eps,None),(None,None),(None,None),(eps,None)]).x
    a0 = np.max([a0,eps])
    a1 = np.max([a1,eps])
    gamma = np.min([np.max([gamma,eps]),1.2]) # artificial lower/upper bound
    return a0,a1,b0,b1,gamma



In [3]:
sequence_length = 30
dt = 1/250
learning_rate = 0.005
batch_size = 10000
batch_num= 20
epoch_num = 300
T = dt * sequence_length

In [4]:
# import yfinance as yf
# # Stock tickers you're interested in
# tickers = [
#     'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'BRK-B'
# ]

# # Define date range
# start_date = '2008-09-26'
# end_date = '2021-09-30'

# # Fetching Close prices data
# data = yf.download(tickers, start=start_date, end=end_date)['Close']

# # Save to CSV for future reference
# data.to_csv('stocks_close_prices_2008_2021.csv')


# Load the dataset. Generated from the above code from yfinance.
data = pd.read_csv("../Data/stocks_close_prices_2008_2021.csv")



In [5]:
for company_name in ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'BRK-B']:
    print(f"Processing {company_name}...")
    S0 = data[company_name].values[2880]
    comapny_price = data[company_name].values

    #Create lists for the parameters
    list_a0 = []
    list_a1 = []
    list_b0 = []
    list_b1 = []
    list_gamma = []
    # list_a0_rescaled = []
    # list_a1_rescaled = []
    # list_b0_rescaled = []
    # list_b1_rescaled = []
    #Compute optimal parameters
    for i in tqdm(np.arange(279,2880,100)): # until 9 March 2020
        x = np.array(data[company_name].iloc[(i-250):i])
        a0,a1,b0,b1,gamma = compute_max_parameters(x)
        
        list_a0 += [a0]
        list_a1 += [a1]
        list_b0 += [b0]
        list_b1 += [b1]
        list_gamma += [gamma]
    a0_fix = [list_a0[26], list_a0[26]]
    a1_fix = [list_a1[26], list_a1[26]]
    b0_fix = [list_b0[26], list_b0[26]]
    b1_fix = [list_b1[26], list_b1[26]]
    gamma_fix = [list_gamma[26], list_gamma[26]]
    a0_robust = [min(list_a0), max(list_a0)]
    a1_robust = [min(list_a1), max(list_a1)]
    b0_robust = [min(list_b0), max(list_b0)]
    b1_robust = [min(list_b1), max(list_b1)]
    gamma_robust = [min(list_gamma), max(list_gamma)]
    parameters = {'S0':[S0,S0],'a0_fix': a0_fix, 'a1_fix': a1_fix, 'b0_fix': b0_fix, 'b1_fix': b1_fix, 'gamma_fix': gamma_fix, 'a0_robust': a0_robust, 'a1_robust': a1_robust, 'b0_robust': b0_robust, 'b1_robust': b1_robust, 'gamma_robust': gamma_robust}
    for para_name, para_value in parameters.items():
        print(f"{para_name}\t{para_value[0]}\t{para_value[1]}")

    # Define the path generator function
    generator_fix = path_generator_GAD(
        time_steps=sequence_length,
        S0=S0,
        a0=a0_fix,
        a1=a1_fix,
        b0=b0_fix,
        b1=b1_fix,
        gamma=gamma_fix,
        dt=dt
    )
    generator_robust = path_generator_GAD(
        time_steps=sequence_length,
        S0=S0,
        a0=a0_robust,
        a1=a1_robust,
        b0=b0_robust,
        b1=b1_robust,
        gamma=gamma_robust,
        dt=dt
    )
    # Generate the paths and save them
    folder_path = f'../DATA/GAD_{company_name}'
    os.makedirs(folder_path, exist_ok=True)
    price_fix_train = generator_fix.generate(100000)/ S0 * 10
    torch.save(price_fix_train, folder_path+'/GAD_fix_train.pt')
    price_fix_test = generator_fix.generate(100000)/ S0 * 10
    torch.save(price_fix_test, folder_path+'/GAD_fix_test.pt')
    price_fix_val = generator_fix.generate(100000)/ S0 * 10
    torch.save(price_fix_val, folder_path+'/GAD_fix_val.pt')
    price_robust_train = generator_robust.generate(100000)/ S0 * 10
    torch.save(price_robust_train, folder_path+'/GAD_robust_train.pt')
    price_robust_test = generator_robust.generate(100000)/ S0 * 10
    torch.save(price_robust_test, folder_path+'/GAD_robust_test.pt')
    price_robust_val = generator_robust.generate(100000)/ S0 * 10
    torch.save(price_robust_val, folder_path+'/GAD_robust_val.pt')
    #check

    print("Fix train data shape:", price_fix_train.shape)
    print("Fix test data shape:", price_fix_test.shape)
    print("Fix val data shape:", price_fix_val.shape)
    print("Robust train data shape:", price_robust_train.shape)
    print("Robust test data shape:", price_robust_test.shape)
    print("Robust val data shape:", price_robust_val.shape)
    print(' ')

Processing AAPL...


100%|██████████| 27/27 [00:11<00:00,  2.45it/s]


S0	64.50927734375	64.50927734375
a0_fix	0.08026119539672945	0.08026119539672945
a1_fix	0.15448404982831093	0.15448404982831093
b0_fix	-0.014462515420704695	-0.014462515420704695
b1_fix	0.2915249538571112	0.2915249538571112
gamma_fix	1.2	1.2
a0_robust	1e-15	1.1851471299961105
a1_robust	0.15448404982831093	1.1842066515407796
b0_robust	-0.014462515420704695	0.27070806574612005
b1_robust	-0.16462619582713453	0.6728123156916942
gamma_robust	0.1765244825592717	1.2
Fix train data shape: torch.Size([100000, 31])
Fix test data shape: torch.Size([100000, 31])
Fix val data shape: torch.Size([100000, 31])
Robust train data shape: torch.Size([100000, 31])
Robust test data shape: torch.Size([100000, 31])
Robust val data shape: torch.Size([100000, 31])
 
Processing MSFT...


100%|██████████| 27/27 [00:11<00:00,  2.37it/s]


S0	143.90606689453125	143.90606689453125
a0_fix	0.08182575836687038	0.08182575836687038
a1_fix	0.1139912502304919	0.1139912502304919
b0_fix	0.0231042587594889	0.0231042587594889
b1_fix	0.20580497491391983	0.20580497491391983
gamma_fix	1.2	1.2
a0_robust	0.00023165210663758613	0.3625759807642315
a1_robust	0.10553282599863656	1.3188820389271954
b0_robust	-0.022856224799609404	0.03862183215824273
b1_robust	-0.06261100207219218	0.413927821213
gamma_robust	0.5013682246049169	1.2
Fix train data shape: torch.Size([100000, 31])
Fix test data shape: torch.Size([100000, 31])
Fix val data shape: torch.Size([100000, 31])
Robust train data shape: torch.Size([100000, 31])
Robust test data shape: torch.Size([100000, 31])
Robust val data shape: torch.Size([100000, 31])
 
Processing AMZN...


...

S0	90.03050231933594	90.03050231933594
a0_fix	0.10654725472207793	0.10654725472207793
a1_fix	0.21272781634615548	0.21272781634615548
b0_fix	0.008792616957013218	0.008792616957013218
b1_fix	0.13562193445019474	0.13562193445019474
gamma_fix	1.018435486798381	1.018435486798381
a0_robust	1e-15	0.8763988976524102
a1_robust	0.165124090192585	1.4992045227822806
b0_robust	-0.011587866241184616	0.3214130096051824
b1_robust	-0.11780753068144421	0.7931290478629783
gamma_robust	0.38074931430598125	1.2
Fix train data shape: torch.Size([100000, 31])
Fix test data shape: torch.Size([100000, 31])
Fix val data shape: torch.Size([100000, 31])
Robust train data shape: torch.Size([100000, 31])
Robust test data shape: torch.Size([100000, 31])
Robust val data shape: torch.Size([100000, 31])
 
Processing GOOGL...


100%|██████████| 27/27 [00:11<00:00,  2.34it/s]


S0	60.50025177001953	60.50025177001953
a0_fix	0.11004173373689423	0.11004173373689423
a1_fix	0.32049868397994835	0.32049868397994835
b0_fix	0.004605484718899998	0.004605484718899998
b1_fix	0.10479782812693487	0.10479782812693487
gamma_fix	0.9156718194603061	0.9156718194603061
a0_robust	0.04594150169328198	0.6747161680356123
a1_robust	0.12847662599292423	1.7099383618945059
b0_robust	-0.012866665026758197	0.0800811055812128
b1_robust	-0.08070666707253986	0.45430327724355374
gamma_robust	0.4234704643961969	1.2
Fix train data shape: torch.Size([100000, 31])
Fix test data shape: torch.Size([100000, 31])
Fix val data shape: torch.Size([100000, 31])
Robust train data shape: torch.Size([100000, 31])
Robust test data shape: torch.Size([100000, 31])
Robust val data shape: torch.Size([100000, 31])
 
Processing BRK-B...


100%|██████████| 27/27 [00:10<00:00,  2.59it/s]


S0	193.1300048828125	193.1300048828125
a0_fix	0.11067534574516467	0.11067534574516467
a1_fix	0.27723654332855396	0.27723654332855396
b0_fix	0.012818934974585396	0.012818934974585396
b1_fix	0.03267397601340038	0.03267397601340038
gamma_fix	0.8772548603753774	0.8772548603753774
a0_robust	0.0715520967575434	0.16720753460375748
a1_robust	0.08892938169655949	0.966085683753249
b0_robust	-0.002967123170458792	0.02345597988122451
b1_robust	-0.026932734755442698	0.3522729726507892
gamma_robust	0.687712516163207	1.19075938348246
Fix train data shape: torch.Size([100000, 31])
Fix test data shape: torch.Size([100000, 31])
Fix val data shape: torch.Size([100000, 31])
Robust train data shape: torch.Size([100000, 31])
Robust test data shape: torch.Size([100000, 31])
Robust val data shape: torch.Size([100000, 31])
 


In [6]:
for company_name in ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'BRK-B']:
    print(f"Processing {company_name} for real test data...")
    real_data_tensor = torch.Tensor(data[company_name])

    price_real_test = torch.Tensor([])
    for start in range(2880, 3180):
        end = start + 31
        part_normalized = real_data_tensor[start:end]/ real_data_tensor[start]*10
        price_real_test = torch.cat((price_real_test, part_normalized.unsqueeze(0)), dim=0)

    print(price_real_test.shape)
    torch.save(price_real_test, f'../Data/GAD_{company_name}/GAD_real_test.pt')
    

Processing AAPL for real test data...
torch.Size([300, 31])
Processing MSFT for real test data...
torch.Size([300, 31])
Processing AMZN for real test data...
torch.Size([300, 31])
Processing GOOGL for real test data...
torch.Size([300, 31])
Processing BRK-B for real test data...
torch.Size([300, 31])
